In [12]:
import sys
import os
import spacy
sys.path.append(os.path.abspath(os.path.join('..')))
from src.data_cleaning import clean_nonprofit_data
from src.nlp_pipeline import lemmatization, get_dominant_topic, map_topics_to_sectors
from gensim.corpora import Dictionary
from gensim.models import LdaModel

Using mission descriptions provided in IRS990 dataset to categorize nonprofits

In [ ]:

nonprofits = clean_nonprofit_data('../data/all_va.parquet')

# Handle the "Unclassified" data gap 
# We only train the model on organizations that provided a mission description
train_df = nonprofits.dropna(subset=["IRS990_ActivityOrMissionDesc"]).copy()
lemmatized_train = lemmatization(train_df["IRS990_ActivityOrMissionDesc"])
tokenized_train = [text.split() for text in lemmatized_train]
-
# Creating the Dictionary and Corpus required for LDA
dictionary = Dictionary(tokenized_train)
dictionary.filter_extremes(no_below=5, no_above=.5)
corpus = [dictionary.doc2bow(text) for text in tokenized_train]

# Train the model with T=14 topics (optimized via coherence scores)
lda_model = LdaModel(
    corpus=corpus, 
    id2word=dictionary, 
    num_topics=14, 
    random_state=42, 
    passes=10
)

# Assign Topic IDs back to the training set
train_df['Dominant_Topic_ID'] = get_dominant_topic(lda_model, corpus, tokenized_train)

# Use your manual mapping to finalize categories for the report 
train_df['Final_Sector_Category'] = map_topics_to_sectors(train_df['Dominant_Topic_ID'])

# This section identifies the largest categorized sectors like Education and Religion 
print(train_df['Final_Sector_Category'].value_counts())

Final_Sector_Category
Public_Benefit    4721
Human_Services    2235
Other             1734
Education         1657
Health             508
Name: count, dtype: int64
